In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from neuralhydrology.evaluation.metrics import calculate_metrics

In [2]:
# ------------------- Paths -------------------
RUN_DIR = Path("./runs")

ensemble_metrics_dir=Path("./ensemble_testing_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [3]:
# run_pattern = "precip_prcp_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
# run_pattern = "precip_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
run_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"


matched_paths = sorted(RUN_DIR.glob(f"{run_pattern}/test/model_epoch030/test_results.p"))
print(f"Found {len(matched_paths)} runs: {[p.parts[-4] for p in matched_paths]}")

Found 8 runs: ['precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2904_123501', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_222_2904_124105', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_333_2904_124709', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_444_2904_125314', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_555_2904_125917', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_666_2904_130523', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_777_2904_131125', 'precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_888_2904_131729']


In [4]:
# Load all runs
all_runs_data = []
for file_path in matched_paths:
    with open(file_path, "rb") as f:
        all_runs_data.append(pickle.load(f))

# Average the simulated flows across seeds, per basin
ensemble_data = {}

for basin_id in all_runs_data[0].keys():
    # Stack simulated flows from all seeds: shape (n_seeds, n_timesteps, time_step)
    sims = np.stack([
        run[basin_id]['1D']['xr']['QObs_mm_d_sim'].values #['QObs(mm/d)_sim'].values 
        for run in all_runs_data
    ], axis=0)
    
    mean_sim = np.mean(sims, axis=0)  # Average across seeds
    
    # Copy structure from first run, replace sim with ensemble mean
    xr_ensemble = all_runs_data[0][basin_id]['1D']['xr'].copy(deep=True)
    xr_ensemble['QObs_mm_d_sim'].values[:] = mean_sim #['QObs(mm/d)_sim'].values[:] = mean_sim #
    
    ensemble_data[basin_id] = {'1D': {'xr': xr_ensemble}}

ensemble_data

{'CAMELS_UY_10': {'1D': {'xr': <xarray.Dataset> Size: 35kB
   Dimensions:        (date: 2191, time_step: 1)
   Coordinates:
     * date           (date) datetime64[ns] 18kB 2008-10-01 ... 2014-09-30
     * time_step      (time_step) int64 8B 0
   Data variables:
       QObs_mm_d_obs  (date, time_step) float32 9kB 0.2376 0.2216 ... 1.265 1.686
       QObs_mm_d_sim  (date, time_step) float32 9kB 0.2122 0.2132 ... 0.5629 0.5599}},
 'CAMELS_UY_11': {'1D': {'xr': <xarray.Dataset> Size: 35kB
   Dimensions:        (date: 2191, time_step: 1)
   Coordinates:
     * date           (date) datetime64[ns] 18kB 2008-10-01 ... 2014-09-30
     * time_step      (time_step) int64 8B 0
   Data variables:
       QObs_mm_d_obs  (date, time_step) float32 9kB 0.3589 0.3589 ... 0.8041 0.6947
       QObs_mm_d_sim  (date, time_step) float32 9kB 0.4802 0.4038 ... 0.8334 0.6516}},
 'CAMELS_UY_15': {'1D': {'xr': <xarray.Dataset> Size: 35kB
   Dimensions:        (date: 2191, time_step: 1)
   Coordinates:
     * dat

In [5]:
# Now compute metrics on the ensemble mean
all_metric_names = [
    'NSE', 'MSE', 'RMSE', 'KGE', 'Alpha-NSE', 'Pearson-r',
    'Beta-KGE', 'Beta-NSE', 'FHV', 'FMS', 'FLV',
    'Peak-Timing', 'Missed-Peaks', 'Peak-MAPE'
]

all_metrics = {}

for basin_id, basin_data in ensemble_data.items():
    xr_ds = basin_data['1D']['xr'].isel(time_step=0)
    
    obs = xr_ds['QObs_mm_d_obs']
    sim = xr_ds['QObs_mm_d_sim']
    
    # Skip basin if all obs or sim are NaN
    if obs.isnull().all() or sim.isnull().all():
        print(f"Skipping {basin_id} — all observed/simulated values are NaN")
        continue
    
    all_metrics[basin_id] = calculate_metrics(
        obs=obs,
        sim=sim,
        metrics=all_metric_names,
        resolution="1D",
        datetime_coord="date"
    )

df_metrics = pd.DataFrame(all_metrics).T
df_metrics.index.name = 'basin_id'
df_metrics

Skipping CAMELS_UY_15 — all observed/simulated values are NaN


,NSE,MSE,RMSE,KGE,Alpha-NSE,Pearson-r,Beta-KGE,Beta-NSE,FHV,FMS,FLV,Peak-Timing,Missed-Peaks,Peak-MAPE
basin_id,,,,,,,,,,,,,,
CAMELS_UY_10,0.714594,0.509399,0.713722,0.856723,1.019783,0.860394,0.974560,-0.016726,6.327100,-0.869644,-335.768402,1.833333,0.550000,34.690292
CAMELS_UY_11,0.584533,2.908295,1.705372,0.630132,0.725604,0.766721,0.915796,-0.040472,-29.509556,26.621759,-39.690739,0.800000,0.548387,58.077316
CAMELS_UY_16,0.742985,3.515949,1.875086,0.825912,0.978990,0.869492,1.113282,0.032271,-0.386656,2.766403,-1165.306396,0.250000,0.294118,44.537094
CAMELS_UY_2,0.354558,2.393443,1.547076,0.398634,1.400995,0.846905,1.421195,0.235952,41.829563,-2.103758,-1104.126221,1.777778,0.800000,40.771564
CAMELS_UY_3,0.777132,1.302128,1.141108,0.835489,1.039152,0.895076,1.120507,0.057192,10.690189,-7.684658,-71.389389,0.545455,0.535714,25.857380
CAMELS_UY_5,0.697023,3.978308,1.994570,0.765078,1.121635,0.873382,1.156081,0.064367,16.364153,2.797476,-588.360901,1.000000,0.640000,32.881985
CAMELS_UY_6,0.834678,0.665504,0.815784,0.834757,1.035258,0.923712,1.142276,0.078250,3.954318,-6.652887,4.563808,1.181818,0.692308,24.748899
CAMELS_UY_7,0.674570,4.239969,2.059118,0.804669,1.131869,0.864156,1.048076,0.022922,15.156670,-21.391573,-294.732330,0.777778,0.434783,46.085487
CAMELS_UY_8,0.682448,3.879352,1.969607,0.840840,1.000097,0.841249,0.988602,-0.004398,3.309579,-9.767051,-54.611622,1.000000,0.428571,42.863197


In [6]:
save_name = run_pattern.split("_seed")[0]
df_metrics.to_csv(ensemble_metrics_dir/f"{save_name}.csv")

In [7]:
df_metrics.median()

NSE               0.705809
MSE               2.650869
RMSE              1.626224
KGE               0.830334
Alpha-NSE         1.027520
Pearson-r         0.866824
Beta-KGE          1.080679
Beta-NSE          0.027596
FHV               5.140709
FMS              -4.378323
FLV            -202.400917
Peak-Timing       1.000000
Missed-Peaks      0.549194
Peak-MAPE        37.730928
dtype: float64